In [23]:
import duckdb
import pandas as pd
import sys, pathlib

sys.path.insert(0, str(pathlib.Path('../src').resolve()))
import irp.config as _config

cfg  = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB   = str((_ROOT / cfg['store']['db_path']).resolve())

def q(sql, params=None):
    with duckdb.connect(DB, read_only=True) as con:
        return con.execute(sql, params or []).df()

print('DB:', DB)

DB: /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


In [26]:
df = q('SELECT * FROM prices')
print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(45548912, 10)


,ticker,source_id,source,date,open,high,low,close,volume,inserted_at
0,5YUSY,5yusy.b,stooq,1999-09-07,5.822,5.822,5.822,5.822,0.0,2026-04-29 22:51:01.500638+02:00
1,5YUSY,5yusy.b,stooq,1999-11-17,5.977,5.977,5.977,5.977,0.0,2026-04-29 22:51:01.500638+02:00
2,5YUSY,5yusy.b,stooq,1999-12-03,6.074,6.074,6.074,6.074,0.0,2026-04-29 22:51:01.500638+02:00
3,5YUSY,5yusy.b,stooq,2000-04-05,6.133,6.133,6.133,6.133,0.0,2026-04-29 22:51:01.500638+02:00
4,5YUSY,5yusy.b,stooq,2000-04-07,6.174,6.174,6.174,6.174,0.0,2026-04-29 22:51:01.500638+02:00


In [27]:
mask =df['source_id'].str[-2:] == '.m'
print(mask.sum())
df[mask]

0


,ticker,source_id,source,date,open,high,low,close,volume,inserted_at


In [13]:
df['source_id'].str[-2:]

0           .b
1           .b
2           .b
3           .b
4           .b
            ..
45527581    us
45527582    us
45527583    us
45527584    us
45527585    us
Name: source_id, Length: 45527586, dtype: str

In [ ]:
df.dtypes

In [4]:
# single ticker
tic = 'MSFT'
q(f"SELECT * FROM prices WHERE ticker = '{tic}' ORDER BY date")


,ticker,source_id,source,date,open,high,low,close,volume,inserted_at
0,MSFT,msft.us,stooq,1986-03-13,0.061571,0.069028,0.061571,0.069028,1.496329e+09,2026-04-29 21:31:41.671653+02:00
1,MSFT,msft.us,stooq,1986-03-14,0.069028,0.069028,0.069028,0.069028,4.469023e+08,2026-04-29 21:31:41.671653+02:00
2,MSFT,msft.us,stooq,1986-03-17,0.069028,0.069028,0.069028,0.069028,1.931286e+08,2026-04-29 21:31:41.671653+02:00
3,MSFT,msft.us,stooq,1986-03-18,0.069028,0.069028,0.069028,0.069028,9.827675e+07,2026-04-29 21:31:41.671653+02:00
4,MSFT,msft.us,stooq,1986-03-19,0.069028,0.069028,0.069028,0.069028,6.945781e+07,2026-04-29 21:31:41.671653+02:00
...,...,...,...,...,...,...,...,...,...,...
10101,MSFT,msft.us,stooq,2026-04-21,420.240000,427.180000,417.200000,424.160000,3.204850e+07,2026-04-29 22:08:29.573210+02:00
10102,MSFT,msft.us,stooq,2026-04-22,426.185000,433.700000,423.670000,432.920000,2.937817e+07,2026-04-29 22:08:29.573210+02:00
10103,MSFT,msft.us,stooq,2026-04-23,419.885000,423.660000,411.410100,415.750000,3.830796e+07,2026-04-29 22:08:29.573210+02:00
10104,MSFT,msft.us,stooq,2026-04-24,416.970000,424.950000,415.800000,424.620000,2.745740e+07,2026-04-29 22:08:29.573210+02:00


In [ ]:
coverage = q("""
    SELECT ticker, source_id, MIN(date) AS first_date, MAX(date) AS last_date, COUNT(*) AS n_rows
    FROM prices
    GROUP BY ticker, source_id
    ORDER BY ticker
""")
coverage

In [ ]:
coverage.sort_values('first_date').head(10)


In [17]:
# Delete MSFT quotes from 2026-04-17 onwards (to test upsert re-insert)
with duckdb.connect(DB) as con:
    deleted = con.execute(
        "DELETE FROM prices WHERE date >= '2026-04-16'"
    ).rowcount
print(f"Deleted {deleted} rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Deleted -1 rows


In [ ]:
mask = coverage['last_date'] >= '2026-04-24'
coverage[mask].sort_values('last_date')


In [ ]:
coverage[coverage['n_rows']<=5]
